# Claim-Amount model — 2026 re-run

This notebook re-runs the **2025 winning severity (claim-amount) model**
([CAA / ENS Challenge Data #161](https://challengedata.ens.fr/challenges/161))
with the **current AutoCarver** (see `pyproject.toml` for the pinned floor).
Companion to `frequency_model_2026.ipynb`.

Everything before the carving step — data loading, the `CM` target, the
stratified split, feature engineering and the column typing — is **identical to
`amount_model.ipynb`** (cells reused verbatim) so the only thing that changes is
the AutoCarver step. That keeps the comparison honest.

**What changes in 2026:**
1. **`ProcessingConfig(n_jobs=...)`** — per-feature carving runs across a process
   pool. Timed below.
2. **`min_freq_alpha`** — bin frequencies are gated by a **Wilson-score CI**, so
   thin bins are merged on statistical grounds (relevant for the heavy-tailed
   amount target).
3. **Selectors** take a `SelectionConfig` and a single `n_best_features` **total**
   budget, split evenly across feature types.
4. **Observation weights** come from the **2026 frequency model's** absolute
   error (exported by `frequency_model_2026.ipynb`), not the 2025 one.

The carver itself stays `ContinuousCarver` — the amount target is continuous,
so there is no `OrdinalCarver`-style upgrade on this side; the Kruskal-Wallis
carving criterion is unchanged from 2025.

## Loading data & target  *(reused from 2025)*

In [1]:
import pandas as pd

data_path = "../data/"

# loading x_train
data = pd.read_csv(data_path + "train_input_Z61KlZo.csv", low_memory=False)
data.set_index("ID", inplace=True)
print("x_train", data.shape)

# loading target
target = pd.read_csv(data_path + "train_output_DzPxaPY.csv", low_memory=False)
target.set_index("ID", inplace=True)
print("y_train", target.shape)

# joining x_train and y_train
data = data.join(target.drop("ANNEE_ASSURANCE", axis=1))
print("data", data.shape)

x_train (383610, 373)
y_train (383610, 4)


data (383610, 376)


## Stratified sampling  *(reused from 2025)*

Stratification uses the discretized claim amount (the saved target carver) so
train and dev share the same amount distribution.


In [2]:
from pathlib import Path

from AutoCarver import BinaryCarver
from sklearn.model_selection import train_test_split

target_col = "CM"

# loading target carver -- re-fitted by frequency_model_2026.ipynb
# (feature = CM, target = frequency), saved light-mode. Same recipe as 2025.
target_carver = BinaryCarver.load(Path("model/cm_carver_freq_tschuprowt_2026.json"))

# discretizing the target
y_freq = target_carver.transform(data)[target_col]

# Train-test split
x_train, x_dev, y_train, y_dev = train_test_split(
    data, data[target_col], test_size=0.2, random_state=42, stratify=y_freq
)
print("y_train mean", y_train.mean())
print("y_dev mean", y_dev.mean())

y_train mean 182.78860026567781
y_dev mean 181.4537592998097


## Feature engineering  *(reused from 2025)*

In [3]:
import warnings
from utils.data_toolkit import Processor

warnings.simplefilter(action="ignore", category=FutureWarning)

proc = Processor()
x_train = proc.fit_transform(x_train)
x_dev = proc.transform(x_dev)
data = proc.transform(data)
data.shape[0], x_train.shape[0], x_dev.shape[0]

(383610, 306888, 76722)

## Column typing  *(reused from 2025)*

In [4]:
from AutoCarver import Features

# sorting columns per type
features = Features.from_dataframe(x_train)

# getting ordinal columns
ordinals = [
    "NB_CASERNES",
    "BDTOPO_BAT_MAX_HAUTEUR",
    "HAUTEUR_MAX",
    "HAUTEUR",
    "BDTOPO_BAT_MAX_HAUTEUR_MAX",
    "MEN_SURF",
    "IND_SNV",
    "IND_INC",
    "IND_Y9",
    "IND_0_Y1",
    "IND",
    "LOG_SOC",
    "LOG_INC",
    "LOG_APA3",
    "LOG_AVA1",
    "MEN_MAIS",
    "MEN_COLL",
    "MEN_FMP",
    "MEN_PROP",
    "MEN_PAUV",
    "MEN",
    "COEFASS",
]
ordinals += [
    "DISTANCE_111",
    "DISTANCE_112",
    "DISTANCE_121",
    "DISTANCE_122",
    "DISTANCE_123",
    "DISTANCE_124",
    "DISTANCE_131",
    "DISTANCE_132",
    "DISTANCE_133",
    "DISTANCE_141",
    "DISTANCE_142",
    "DISTANCE_211",
    "DISTANCE_212",
    "DISTANCE_213",
    "DISTANCE_221",
    "DISTANCE_222",
    "DISTANCE_223",
    "DISTANCE_231",
    "DISTANCE_242",
    "DISTANCE_243",
    "DISTANCE_244",
    "DISTANCE_311",
    "DISTANCE_312",
    "DISTANCE_313",
    "DISTANCE_321",
    "DISTANCE_322",
    "DISTANCE_323",
    "DISTANCE_324",
    "DISTANCE_331",
    "DISTANCE_332",
    "DISTANCE_333",
    "DISTANCE_334",
    "DISTANCE_335",
    "DISTANCE_411",
    "DISTANCE_412",
    "DISTANCE_421",
    "DISTANCE_422",
    "DISTANCE_423",
    "DISTANCE_511",
    "DISTANCE_512",
    "DISTANCE_521",
    "DISTANCE_522",
    "DISTANCE_523",
    "PROPORTION_11",
    "PROPORTION_12",
    "PROPORTION_13",
    "PROPORTION_14",
    "PROPORTION_21",
    "PROPORTION_22",
    "PROPORTION_23",
    "PROPORTION_24",
    "PROPORTION_31",
    "PROPORTION_32",
    "PROPORTION_33",
    "PROPORTION_41",
    "PROPORTION_42",
    "PROPORTION_51",
    "PROPORTION_52",
    "MEN_1IND",
    "MEN_5IND",
    "LOG_A1_A2",
    "LOG_A2_A3",
    "IND_Y1_Y2",
    "IND_Y2_Y3",
    "IND_Y3_Y4",
    "IND_Y4_Y5",
    "IND_Y5_Y6",
    "IND_Y6_Y7",
    "IND_Y7_Y8",
    "IND_Y8_Y9",
    "DISTANCE_1",
    "DISTANCE_2",
    "ALTITUDE_1",
    "ALTITUDE_2",
    "ALTITUDE_3",
    "ALTITUDE_4",
    "ALTITUDE_5",
    "NBJTX25_MM_A",
    "NBJTX25_MMAX_A",
    "NBJTX25_MSOM_A",
    "NBJTX0_MM_A",
    "NBJTX0_MMAX_A",
    "NBJTX0_MSOM_A",
    "NBJTXI27_MM_A",
    "NBJTXI27_MMAX_A",
    "NBJTXI27_MSOM_A",
    "NBJTXS32_MM_A",
    "NBJTXS32_MMAX_A",
    "NBJTXS32_MSOM_A",
    "NBJTXI20_MM_A",
    "NBJTXI20_MMAX_A",
    "NBJTXI20_MSOM_A",
    "NBJTX30_MM_A",
    "NBJTX30_MMAX_A",
    "NBJTX30_MSOM_A",
    "NBJTX35_MM_A",
    "NBJTX35_MMAX_A",
    "NBJTX35_MSOM_A",
    "NBJTN10_MM_A",
    "NBJTN10_MMAX_A",
    "NBJTN10_MSOM_A",
    "NBJTNI10_MM_A",
    "NBJTNI10_MMAX_A",
    "NBJTNI10_MSOM_A",
    "NBJTN5_MM_A",
    "NBJTN5_MMAX_A",
    "NBJTN5_MSOM_A",
    "NBJTNS25_MM_A",
    "NBJTNS25_MMAX_A",
    "NBJTNS25_MSOM_A",
    "NBJTNI15_MM_A",
    "NBJTNI15_MMAX_A",
    "NBJTNI15_MSOM_A",
    "NBJTNI20_MM_A",
    "NBJTNI20_MMAX_A",
    "NBJTNI20_MSOM_A",
    "NBJTNS20_MM_A",
    "NBJTNS20_MMAX_A",
    "NBJTNS20_MSOM_A",
    "NBJTMS24_MM_A",
    "NBJTMS24_MMAX_A",
    "NBJTMS24_MSOM_A",
    "TAMPLIAB_VOR_MM_A",
    "TAMPLIAB_VOR_MMAX_A",
    "TAMPLIM_VOR_MM_A",
    "TAMPLIM_VOR_MMAX_A",
    "TM_VOR_MM_A",
    "TM_VOR_MMAX_A",
    "TMM_VOR_MM_A",
    "TMM_VOR_MMAX_A",
    "TMMAX_VOR_MM_A",
    "TMMAX_VOR_MMAX_A",
    "TMMIN_VOR_MM_A",
    "TMMIN_VOR_MMAX_A",
    "TN_VOR_MM_A",
    "TN_VOR_MMAX_A",
    "TNAB_VOR_MM_A",
    "TNAB_VOR_MMAX_A",
    "TNMAX_VOR_MM_A",
    "TNMAX_VOR_MMAX_A",
    "TX_VOR_MM_A",
    "TX_VOR_MMAX_A",
    "TXAB_VOR_MM_A",
    "TXAB_VOR_MMAX_A",
    "TXMIN_VOR_MM_A",
    "TXMIN_VOR_MMAX_A",
    "NBJFF10_MM_A",
    "NBJFF10_MMAX_A",
    "NBJFF10_MSOM_A",
    "NBJFF16_MM_A",
    "NBJFF16_MMAX_A",
    "NBJFF16_MSOM_A",
    "NBJFF28_MM_A",
    "NBJFF28_MMAX_A",
    "NBJFF28_MSOM_A",
    "NBJFXI3S10_MM_A",
    "NBJFXI3S10_MMAX_A",
    "NBJFXI3S10_MSOM_A",
    "NBJFXI3S16_MM_A",
    "NBJFXI3S16_MMAX_A",
    "NBJFXI3S16_MSOM_A",
    "NBJFXI3S28_MM_A",
    "NBJFXI3S28_MMAX_A",
    "NBJFXI3S28_MSOM_A",
    "NBJFXY8_MM_A",
    "NBJFXY8_MMAX_A",
    "NBJFXY8_MSOM_A",
    "NBJFXY10_MM_A",
    "NBJFXY10_MMAX_A",
    "NBJFXY10_MSOM_A",
    "NBJFXY15_MM_A",
    "NBJFXY15_MMAX_A",
    "NBJFXY15_MSOM_A",
    "FFM_VOR_MM_A",
    "FFM_VOR_MMAX_A",
    "FXI3SAB_VOR_MM_A",
    "FXI3SAB_VOR_MMAX_A",
    "FXIAB_VOR_MM_A",
    "FXIAB_VOR_MMAX_A",
    "FXYAB_VOR_MM_A",
    "FXYAB_VOR_MMAX_A",
    "FFM_VOR_COM_MM_A_Y",
    "FFM_VOR_COM_MMAX_A_Y",
    "FXI3SAB_VOR_COM_MM_A_Y",
    "FXI3SAB_VOR_COM_MMAX_A_Y",
    "NBJRR50_MM_A",
    "NBJRR50_MMAX_A",
    "NBJRR50_MSOM_A",
    "NBJRR1_MM_A",
    "NBJRR1_MMAX_A",
    "NBJRR1_MSOM_A",
    "NBJRR5_MM_A",
    "NBJRR5_MMAX_A",
    "NBJRR5_MSOM_A",
    "NBJRR10_MM_A",
    "NBJRR10_MMAX_A",
    "NBJRR10_MSOM_A",
    "NBJRR30_MM_A",
    "NBJRR30_MMAX_A",
    "NBJRR30_MSOM_A",
    "NBJRR100_MM_A",
    "NBJRR100_MMAX_A",
    "NBJRR100_MSOM_A",
    "RR_VOR_MM_A",
    "RR_VOR_MMAX_A",
    "RRAB_VOR_MM_A",
    "RRAB_VOR_MMAX_A",
]
ordinals += ["TAILLE1", "TAILLE2"]
ordinal_columns = {
    col: list(data[col].value_counts().sort_index().index)
    for col in ordinals
    if col in data.columns
}
ordinal_columns["PROPORTION_32"] += ["10. > 90"]
ordinal_columns.update(
    {
        "CARACT4": [
            "absence de surface",
            "Surface de moins d",
            "Surface entre 501",
            "Surface entre 1001",
            "Surface entre 1501",
            "Surface de plus de",
        ],
        "SURFACE4": [
            "0",
            "500",
            "1000",
            "1500",
            "2000",
            "2500",
            "3000",
            "3500",
            "4000",
            "4500",
            "5000",
            "5500",
            "6000",
            "6500",
            "7000",
            "7000+",
        ],
        "SURFACE6": [
            "0",
            "500",
            "1000",
            "1500",
            "2000",
            "2500",
            "3000",
            "3500",
            "4000",
            "4500",
            "5000",
            "5500",
            "6000",
            "6500",
            "7000",
            "7000+",
        ],
        "total_surface_2023": [
            "Aucun feu",
            "<10ha",
            "10-20ha",
            "20-50ha",
            "50-100ha",
            "100-200ha",
            ">200ha",
        ],
        "total_surface_5y": [
            "Aucun feu",
            "<10ha",
            "10-20ha",
            "20-50ha",
            "50-100ha",
            "100-200ha",
            ">200ha",
        ],
        "surface_over_forest": [
            "Absence de feu",
            "<0.05",
            "0.05-0.1",
            "0.1-0.2",
            "0.5-2",
        ],
        "fire_extinction_rates": ["Aucun feu", "<50%", "50-70%", "70-85%", ">85%"],
    }
)

# columns that are to be removed (target + no values)
to_remove = target.columns.tolist() + [target_col]
to_remove += [c for c in data.columns if "MMSOM" in c]
to_remove += [
    "DEROG3",
    "DEROG13",
    "DEROG16",
    "DEROG8",
    "DEROG14",
    "TARGET",
    "DEROG13_formatted",
    "DEROG8_formatted",
    "DEROG3_formatted",
    "DEROG16_formatted",
    "DEROG14_formatted",
    "IND_Y1_Y2_num",
    "IND_INC_num",
]

# removing columns
categorical_columns = [
    col.name
    for col in features.categoricals
    if col not in to_remove and col not in ordinal_columns
]
categorical_columns += ["TYPERS"]
numerical_columns = [
    col.name
    for col in features.numericals
    if col not in to_remove
    and col not in ordinal_columns
    and col not in categorical_columns
]
print(
    len(categorical_columns),
    len(numerical_columns),
    len(ordinal_columns),
    len(categorical_columns) + len(numerical_columns) + len(ordinal_columns),
)

193 115 238 546


## 2026 carving — ContinuousCarver + multiprocessing + Wilson-CI

Same carver class and `max_n_mod=5` as 2025 — only the execution config is new.
`min_freq` is left at the 7.6+ default of **0.02** where 2025 carved this model at
**0.03**: a deviation between the two eras, on record in the article's setup footnote. The
lower threshold admits thinner buckets, i.e. *more* work (and `ZONE` survives carving where
it did not at 0.03), so it does not flatter the 2026 wall-clock.

`n_jobs` parallelises the per-feature combination search; set it to the core count on the
competition machine. `min_freq_alpha=0.05` is the 95% Wilson interval used to test each
bin's frequency.

In [5]:
import time

from AutoCarver import ContinuousCarver, Features
from AutoCarver.discretizers import ProcessingConfig

N_JOBS = 6

config = ProcessingConfig(
    dropna=False, copy=False, verbose=False, n_jobs=N_JOBS, min_freq_alpha=0.05
)

qualitatives = Features(categoricals=categorical_columns, ordinals=ordinal_columns)

# max_n_mod is 5 by default, as in 2025; min_freq is left at the 7.6+ default (0.02)
# where 2025 carved this model at 0.03 -- a deviation on record in the setup footnote
carver = ContinuousCarver(features=qualitatives, config=config)

t0 = time.perf_counter()
x_train = carver.fit_transform(x_train, y_train, X_dev=x_dev, y_dev=y_dev)
print(
    f"[2026] qualitative carving: {time.perf_counter() - t0:.1f}s on {N_JOBS} workers"
)
carver.summary

[ContinuousCarver] Carving:   0%|          | 0/431 [00:00<?, ?feature/s]

[ContinuousCarver] dropped 1/431 feature(s) (no robust train/dev combination): INDEM1


[2026] qualitative carving: 139.8s on 6 workers


content  \
feature                          kruskal    n_mod label                                                    
Categorical('ACTIVIT2')          159.526506 3.0   0      [ACT2, ACT8, ACT4, ACT6, ACT7, __OTHER__, ACT9]   
                                                  1                                                 ACT5   
                                                  2                                         [ACT3, ACT1]   
Categorical('VOCATION')          507.252150 3.0   0            [VOC7, VOC8, VOC5, VOC3, VOC2, __OTHER__]   
                                                  1                                         [VOC4, VOC1]   
...                                                                                                  ...   
Ordinal('surface_over_forest')   10.096102  4.0   3                                     [0.5-2, 0.1-0.2]   
Ordinal('fire_extinction_rates') 2.238349   2.0   0                    [<50%, 50-70%, 70-85%, Aucun feu]   
                                                  1                                                 >85%   
Categorical('INDEM1')            NaN        NaN   O                                                    O   
                                                  N                                                    N   

                                                         target_mean  \
feature                          kruskal    n_mod label                
Categorical('ACTIVIT2')          159.526506 3.0   0        86.255102   
                                                  1       165.349191   
                                                  2       194.858259   
Categorical('VOCATION')          507.252150 3.0   0        58.531347   
                                                  1       116.736895   
...                                                              ...   
Ordinal('surface_over_forest')   10.096102  4.0   3       196.900856   
Ordinal('fire_extinction_rates') 2.238349   2.0   0       161.468501   
                                                  1       244.639328   
Categorical('INDEM1')            NaN        NaN   O       150.789799   
                                                  N       184.348272   

                                                         frequency     count  \
feature                          kruskal    n_mod label                        
Categorical('ACTIVIT2')          159.526506 3.0   0       0.073017   22408.0   
                                                  1       0.140289   43053.0   
                                                  2       0.786694  241427.0   
Categorical('VOCATION')          507.252150 3.0   0       0.285212   87528.0   
                                                  1       0.077422   23760.0   
...                                                            ...       ...   
Ordinal('surface_over_forest')   10.096102  4.0   3       0.054733   16797.0   
Ordinal('fire_extinction_rates') 2.238349   2.0   0       0.743659  228220.0   
                                                  1       0.256341   78668.0   
Categorical('INDEM1')            NaN        NaN   O       0.046476   14263.0   
                                                  N       0.953524  292625.0   

                                                                 std  dropped  \
feature                          kruskal    n_mod label                         
Categorical('ACTIVIT2')          159.526506 3.0   0      4147.474384    False   
                                                  1      5811.476729    False   
                                                  2      7056.154998    False   
Categorical('VOCATION')          507.252150 3.0   0      3653.991661    False   
                                                  1      5004.026018    False   
...                                                              ...      ...   
Ordinal('surface_over_forest')   10.096102  4.0   3      7644.9966

No processing for continuous features

In [6]:
quantitatives = Features(numericals=numerical_columns)

### Apply carver to dev + OOS

In [7]:
x_dev = carver.transform(x_dev)

oos = pd.read_csv(data_path + "test_input_5qJzHrr.csv", low_memory=False)
oos = proc.transform(oos)
oos = carver.transform(oos)
oos.set_index("ID", inplace=True)
oos.shape[0], x_train.shape[0], x_dev.shape[0]

(95852, 306888, 76722)

## Frequency-model predictions & error weights  *(2026 frequency run)*

As in 2025, each observation is weighted by the **absolute error of the
frequency model** so the amount model concentrates on the first stage's
mistakes. The difference: predictions come from `frequency_model_2026.ipynb`,
which writes `data/frequency_2026.csv` (train + dev, carrying the `ID` index)
and `data/oos_frequency_2026.csv`. **Run that notebook first** — these files are
this notebook's inputs, and a stale pair describes a frequency model that no
longer exists.

In [8]:
FREQ_VERSION = "2026"

# frequency predictions on train + dev
merged = pd.read_csv(data_path + f"frequency_{FREQ_VERSION}.csv", low_memory=False)
merged.set_index("ID", inplace=True)

x_train = x_train.join(merged[[c for c in merged.columns if c not in x_train.columns]])
x_dev = x_dev.join(merged[[c for c in merged.columns if c not in x_dev.columns]])

# frequency predictions on OOS
oos_freq = pd.read_csv(
    data_path + f"oos_frequency_{FREQ_VERSION}.csv", low_memory=False
)
oos_freq.set_index("ID", inplace=True)

oos = oos.join(oos_freq[[c for c in oos_freq.columns if c not in oos.columns]])
x_train.shape[0], x_dev.shape[0], oos.shape[0]

(306888, 76722, 95852)

In [9]:
# weight = absolute error of the frequency model (boosting-flavoured hand-off)
w_train = (x_train["pred_sum"] - x_train["FREQ"]).abs()
w_dev = (x_dev["pred_sum"] - x_dev["FREQ"]).abs()

## Feature selection  *(single selector, one config)*

One `RegressionSelector` over **both** feature types with a single `SelectionConfig`,
instead of one selector per type. `n_best_features` is a **total** budget, split **evenly**
across feature types, so the 200 below buys 100 carved qualitative + 100 raw quantitative
features.

Worth knowing: the frequency and severity targets want **opposite** feature mixes. Tilting
the budget towards carved qualitatives helps this arm and hurts the frequency arm, so an
even split is a compromise rather than an optimum on either side.

Measures are `DistanceMeasure` + `SpearmanFilter` on quantitatives and the library-default
Kruskal association on carved qualitatives, with a Cramer's V 0.9 redundancy cut.

In [10]:
from AutoCarver.selectors import (
    CramervFilter,
    DistanceMeasure,
    RegressionSelector,
    SelectionConfig,
    SpearmanFilter,
)

config = SelectionConfig(
    quantitative_measures=[DistanceMeasure(threshold=0.002)],
    quantitative_filters=[SpearmanFilter(threshold=0.9)],
    qualitative_filters=[CramervFilter(threshold=0.9)],
)
selector = RegressionSelector(
    qualitatives + quantitatives, n_best_features=200, config=config
)
selector.fit(x_train, y_train)
print("selected:", len(selector.selected_features))
selector.summary

selected: 200


,feature,Nan,Mode,measure,association,rank,filter,redundancy,redundancy_with,selected
0,Numerical('SURFACE11'),0.228888,0.066145,Distance,-0.032622,0.0,Spearman,0.000000,itself,True
1,Numerical('KAPITAL32'),0.000000,0.348323,Distance,-0.026539,1.0,Spearman,0.632294,SURFACE11,True
2,Numerical('SURFACE17'),0.000000,0.831893,Distance,-0.025193,2.0,Spearman,0.485804,SURFACE11,True
3,Numerical('KAPITAL_MAX'),0.000000,0.248618,Distance,-0.023689,3.0,Spearman,0.824240,KAPITAL32,True
4,Numerical('SURFACE10'),0.019496,0.523396,Distance,-0.020762,4.0,Spearman,0.644371,SURFACE11,True
...,...,...,...,...,...,...,...,...,...,...
540,Ordinal('NBJRR100_MMAX_A'),0.567402,0.370797,KruskalEtaSquared,0.000024,NaN,None,NaN,None,False
541,Ordinal('NBJRR100_MSOM_A'),0.567402,0.376906,KruskalEtaSquared,0.000028,NaN,None,NaN,None,False
542,Ordinal('SURFACE6'),0.000000,0.605025,KruskalEtaSquared,0.002966,NaN,Cramerv,0.999987,SURFACE4,False
543,Ordinal('surface_over_forest'),0.000000,0.721455,KruskalEtaSquared,0.000023,NaN,None,NaN,None,False


In [11]:
best_features = selector.selected_features.names

## XGBoost + Optuna  *(unchanged objectives from `utils.objectives`)*

`reg:tweedie` regressor on the frequency-error-weighted samples, exactly as in
2025 — any dev-metric delta is attributable to the carving step, not the tuner. Drop
`N_TRIALS` while iterating.


In [12]:
import optuna

from utils.objectives import get_regression_objective

N_TRIALS = 400

# tweedie requires a non-negative target
y_transform = lambda u: u.where(u >= 0, 0)

objective = get_regression_objective(
    x_train[best_features],
    y_transform(y_train),
    x_dev[best_features],
    y_transform(y_dev),
    objective="reg:tweedie",
    w_train=w_train,
    w_dev=w_dev,
)
# seeded so the tuning is reproducible: the split and every XGBoost estimator
# already use random_state=42, and carving is deterministic, so this is the last
# source of run-to-run variance.
study = optuna.create_study(
    direction="minimize", sampler=optuna.samplers.TPESampler(seed=42)
)
study.optimize(objective, n_trials=N_TRIALS)
study.best_params

[I 2026-08-31 09:15:07,087] A new study created in memory with name: no-name-7e7f1482-0686-4935-a572-9c23ebe4ec60


C:\Users\defra\Desktop\git\PROJECTS\caa-challenge\.venv\Lib\site-packages\xgboost\core.py:751: UserWarning: [09:15:18] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[I 2026-08-31 09:15:20,296] Trial 0 finished with value: 1.0552441568676813e+34 and parameters: {'objective': 'reg:tweedie', 'n_estimators': 287, 'learning_rate': 0.9507192349792751, 'max_depth': 8, 'min_child_weight': 12, 'subsample': 0.3248149123539492, 'colsample_bytree': 0.32479561626896214, 'colsample_bylevel': 0.24646688973455957, 'gamma': 8.661761457749352, 'alpha': 6.011150117432088, 'lambda': 7.080725777960454, 'tweedie_variance_power': 1.216467595436642}. Best is trial 0 with value: 1.0552441568676813e+34.


[I 2026-08-31 09:15:31,162] Trial 1 finished with value: 3494010.1967561687 and parameters: {'objective': 'reg:tweedie', 'n_estimators': 585, 'learning_rate': 0.8324593965363417, 'max_depth': 3, 'min_child_weight': 4, 'subsample': 0.34672360788274703, 'colsample_bytree': 0.4433937943676302, 'colsample_bylevel': 0.6198051453057902, 'gamma': 4.319450186421157, 'alpha': 2.9122914019804194, 'lambda': 6.118528947223795, 'tweedie_variance_power': 1.3115950885216334}. Best is trial 1 with value: 3494010.1967561687.


[I 2026-08-31 09:15:39,235] Trial 2 finished with value: 7204.383936191784 and parameters: {'objective': 'reg:tweedie', 'n_estimators': 246, 'learning_rate': 0.3664252071093623, 'max_depth': 5, 'min_child_weight': 16, 'subsample': 0.3597390257266878, 'colsample_bytree': 0.6113875507308892, 'colsample_bylevel': 0.6739316550896339, 'gamma': 0.46450412719997725, 'alpha': 6.075448519014383, 'lambda': 1.7052412368729153, 'tweedie_variance_power': 1.2520412743882237}. Best is trial 2 with value: 7204.383936191784.


...

[I 2026-08-31 10:16:53,331] Trial 396 finished with value: 7217.787306228414 and parameters: {'objective': 'reg:tweedie', 'n_estimators': 516, 'learning_rate': 0.20164533801642653, 'max_depth': 1, 'min_child_weight': 18, 'subsample': 0.546639278107269, 'colsample_bytree': 0.8288349292203623, 'colsample_bylevel': 0.9237887703637315, 'gamma': 9.126545106312076, 'alpha': 3.520393483234635, 'lambda': 9.52793944357902, 'tweedie_variance_power': 1.9234346279216334}. Best is trial 329 with value: 6469.6972025111745.


[I 2026-08-31 10:17:03,067] Trial 397 finished with value: 7107.9599524926825 and parameters: {'objective': 'reg:tweedie', 'n_estimators': 551, 'learning_rate': 0.2724346328605292, 'max_depth': 1, 'min_child_weight': 3, 'subsample': 0.564776712112922, 'colsample_bytree': 0.8129201106509075, 'colsample_bylevel': 0.4871555943883151, 'gamma': 1.7447990737630126, 'alpha': 3.0160738712788593, 'lambda': 9.402079314740064, 'tweedie_variance_power': 1.90613695587021}. Best is trial 329 with value: 6469.6972025111745.


[I 2026-08-31 10:17:13,063] Trial 398 finished with value: 6663.860386889451 and parameters: {'objective': 'reg:tweedie', 'n_estimators': 563, 'learning_rate': 0.15640173713142608, 'max_depth': 1, 'min_child_weight': 19, 'subsample': 0.5980614215920356, 'colsample_bytree': 0.7931582683102384, 'colsample_bylevel': 0.3147255582517354, 'gamma': 1.6709607356624983, 'alpha': 3.7112809457664895, 'lambda': 9.232497976263941, 'tweedie_variance_power': 1.9342530631884864}. Best is trial 329 with value: 6469.6972025111745.


[I 2026-08-31 10:17:22,949] Trial 399 finished with value: 6536.4030044101655 and parameters: {'objective': 'reg:tweedie', 'n_estimators': 535, 'learning_rate': 0.1842490626181942, 'max_depth': 1, 'min_child_weight': 19, 'subsample': 0.5792820241338048, 'colsample_bytree': 0.8745706395939803, 'colsample_bylevel': 0.2527485182744681, 'gamma': 1.9781461923725148, 'alpha': 2.7071256615809554, 'lambda': 9.65283325308684, 'tweedie_variance_power': 1.8874053487899527}. Best is trial 329 with value: 6469.6972025111745.


{'objective': 'reg:tweedie',
 'n_estimators': 570,
 'learning_rate': 0.2595525275491146,
 'max_depth': 1,
 'min_child_weight': 18,
 'subsample': 0.6064480180480966,
 'colsample_bytree': 0.8405603469782187,
 'colsample_bylevel': 0.9959895581557628,
 'gamma': 5.529890290582709,
 'alpha': 3.275010657889333,
 'lambda': 9.461731268272077,
 'tweedie_variance_power': 1.8914621343096838}

In [13]:
from utils.objectives import get_best_regression_model

selected_features, xgb = get_best_regression_model(
    study.best_params,
    x_train[best_features],
    y_transform(y_train),
    x_dev[best_features],
    y_transform(y_dev),
    w_train=w_train,
    w_dev=w_dev,
    train_on_full=False,
)

used features: ['KAPITAL40', 'VOCATION', 'TYPERS', 'KAPITAL41', 'KAPITAL42', 'RISK11', 'DEROG12', 'KAPITAL37', 'ACTIVIT2', 'KAPITAL43', 'NBJFXI3S28_MM_A_NBJFXI3S28_MMAX_A', 'DEROG4', 'DEROG5', 'TXAB_VOR_MM_A_TXAB_VOR_MMAX_A', 'TN_VOR_MM_A_TN_VOR_MMAX_A', 'TAMPLIAB_VOR_MM_A_TAMPLIAB_VOR_MMAX_A', 'NBJTX0_MM_A_NBJTX0_MMAX_A', 'INDEM2', 'TX_VOR_MM_A_TX_VOR_MMAX_A', 'RRAB_VOR_MM_A_RRAB_VOR_MMAX_A', 'TAMPLIM_VOR_MM_A_TAMPLIM_VOR_MMAX_A', 'NBJTXI27_MM_A_NBJTXI27_MMAX_A', 'NBJFF28_MM_A_NBJFF28_MMAX_A', 'EQUIPEMENT2', 'RISK12', 'NBJTXI20_MM_A_NBJTXI20_MMAX_A', 'ZONE', 'NBJTX25_MM_A_NBJTX25_MMAX_A', 'TNAB_VOR_MM_A_TNAB_VOR_MMAX_A', 'TXMIN_VOR_MM_A_TXMIN_VOR_MMAX_A', 'RISK6', 'TM_VOR_MM_A_TM_VOR_MMAX_A', 'NBJRR30_MM_A_NBJRR30_MMAX_A', 'TMMAX_VOR_MM_A_TMMAX_VOR_MMAX_A', 'NBJTNI10_MM_A_NBJTNI10_MMAX_A', 'KAPITAL35', 'IND_REGION', 'NBJRR1_MM_A_NBJRR1_MMAX_A', 'TMMIN_VOR_MM_A_TMMIN_VOR_MMAX_A', 'total_surface_crossed', 'RR_VOR_MM_A_RR_VOR_MMAX_A', 'IND_SNV_REGION', 'NBJTXS32_MM_A_NBJTXS32_MMAX_A', 'N

RMSE Train: 6888.927095479314


RMSE Dev:   6469.6972025111745


## Prediction on OOS & challenge submission

In [14]:
# expected cost = frequency x severity
oos["CM"] = xgb.predict(oos[selected_features])

pred = oos.reset_index()[["ID", "ANNEE_ASSURANCE", "pred_sum", "CM"]].rename(
    columns={"pred_sum": "FREQ"}
)
pred["CHARGE"] = pred["FREQ"] * pred["CM"]
# pred.to_csv("predictions/2026_amount_pred.csv", index=False)
pred.head()

,ID,ANNEE_ASSURANCE,FREQ,CM,CHARGE
0,383611,0.813699,0.963831,39.561230,38.130350
1,383612,1.000000,0.694419,91.541489,63.568104
2,383613,0.586301,0.400150,110.509445,44.220392
3,383614,1.000000,0.373611,427.521820,159.726933
4,383615,0.753425,0.495960,322.615326,160.004360


## End-to-end `CHARGE` on train / dev

The challenge metric is `CHARGE`, the **total** claim cost. In the raw target
`CHARGE = CM x (FREQ x ANNEE_ASSURANCE)` -- average claim amount times claim count
(verified on `train_output`: `CHARGE / CM` is exactly `FREQ * ANNEE_ASSURANCE`, an
integer 1..5, on every one of the 2,352 rows with a claim). The frequency model predicts
the claim **count** (`pred_sum`, the `CLASS_MEANS`-weighted expectation), so
`pred_sum * CM_pred` lands on the same scale as the true `CHARGE` -- the same
construction as the OOS submission above and as 2025 (`amount_model.ipynb` cell 65).

**Read the dev number with care.** Optuna selected on this dev set over 400 trials, so
the dev `CHARGE` RMSE is optimistic -- it is a selection-contaminated estimate, not a
held-out one. Train and dev are printed together so the gap is visible.

In [15]:
from sklearn.metrics import root_mean_squared_error

for label, frame in (("Train", x_train), ("Dev", x_dev)):
    # true CHARGE lives in the raw target; fall back to it if the carved frame dropped it
    true_charge = (
        frame["CHARGE"]
        if "CHARGE" in frame.columns
        else target.loc[frame.index, "CHARGE"]
    )
    pred_charge = frame["pred_sum"] * xgb.predict(frame[selected_features])
    print(f"CHARGE RMSE {label}: {root_mean_squared_error(true_charge, pred_charge)}")

CHARGE RMSE Train: 7031.15579812017


CHARGE RMSE Dev: 6482.842161889131
